In [1]:
import pandas as pd

# Load datasets
df = pd.read_csv("processed_dataset.csv")
school_df = pd.read_csv("school_zone.csv")

# Clean CASE_ID
df['CASE_ID'] = df['CASE_ID'].astype(str).str.strip().str.replace(".0", "", regex=False)
school_df['CASE_ID'] = school_df['CASE_ID'].astype(str).str.strip().str.replace(".0", "", regex=False)

# Create set of school CASE_IDs
school_ids = set(school_df['CASE_ID'].drop_duplicates())

# Create indicator column
df['school_zone'] = df['CASE_ID'].isin(school_ids).astype(int)

# 🔍 Check before overwrite
print(df['school_zone'].value_counts())

# df['label'] = (df['SEVERITY'] >= 3).astype(int)
# df = df.drop(columns ='SEVERITY')

# 💾 OVERWRITE original file
df.to_csv("processed_dataset.csv", index=False)

school_zone
0    6349
1    2954
Name: count, dtype: int64


Protocal A cross city OOD 

### Creating splits for Cross City OOD

In [2]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("processed_dataset.csv")

# --- Optional: convert severity to binary if needed ---
# Uncomment if your label is not already binary
# df['label'] = (df['SEVERITY'] >= 3).astype(int)

label_col = 'label'  # or 'label' if you created one
city_col = 'CITY'

# --- Create splits ---
def create_cross_city_splits(df, city_col, label_col, seed=0):
    cities = df[city_col].unique()
    splits = []

    for test_city in cities:
        # --- Test set (one city) ---
        test_idx = df[df[city_col] == test_city].index

        # --- Train + Val ---
        train_val_df = df[df[city_col] != test_city]

        # --- Stratified split ---
        train_idx, val_idx = train_test_split(
            train_val_df.index,
            test_size=0.15,
            stratify=train_val_df[label_col],
            random_state=seed
        )

        splits.append({
            "test_city": str(test_city),
            "train_idx": list(map(int, train_idx)),
            "val_idx": list(map(int, val_idx)),
            "test_idx": list(map(int, test_idx))
        })

    return splits

# --- Generate splits for 5 seeds ---
all_splits = {}

for seed in range(5):
    all_splits[seed] = create_cross_city_splits(df, city_col, label_col, seed)

# --- Save splits ---
os.makedirs("splits", exist_ok=True)

for seed in all_splits:
    for i, split in enumerate(all_splits[seed]):
        filename = f"splits/city_seed{seed}_split{i}.json"
        with open(filename, "w") as f:
            json.dump(split, f)

print("✅ Splits saved successfully!")

✅ Splits saved successfully!


### Creating Temporal OOD Splits

In [3]:
# --- Column name ---
year_col = "ACCIDENT_YEAR"  # based on your dataset

# --- Create split ---
train_idx = df[(df[year_col] >= 2013) & (df[year_col] <= 2019)].index
val_idx   = df[(df[year_col] >= 2020) & (df[year_col] <= 2021)].index
test_idx  = df[(df[year_col] >= 2022) & (df[year_col] <= 2024)].index

split = {
    "train_idx": list(map(int, train_idx)),
    "val_idx": list(map(int, val_idx)),
    "test_idx": list(map(int, test_idx))
}

# --- Save ---
os.makedirs("splits", exist_ok=True)

with open("splits/temporal_split.json", "w") as f:
    json.dump(split, f)

print("✅ Temporal split saved!")

✅ Temporal split saved!


In [4]:
print("Train years:", sorted(df.loc[train_idx]['ACCIDENT_YEAR'].unique()))
print("Val years:", sorted(df.loc[val_idx]['ACCIDENT_YEAR'].unique()))
print("Test years:", sorted(df.loc[test_idx]['ACCIDENT_YEAR'].unique()))

print("\nSizes:")
print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

Train years: [2014, 2015, 2016, 2017, 2018, 2019]
Val years: [2020, 2021]
Test years: [2022, 2023, 2024]

Sizes:
Train: 5173
Val: 1465
Test: 2665


### Creating Policy Shift (School Zones) Splits 

In [5]:
from sklearn.model_selection import train_test_split

# --- Split based on school_zone ---
train_val_df = df[df['school_zone'] == 1]
test_df      = df[df['school_zone'] == 0]

# --- Stratified train/val split ---
train_idx, val_idx = train_test_split(
    train_val_df.index,
    test_size=0.15,
    stratify=train_val_df['label'],
    random_state=0
)

test_idx = test_df.index

# --- Save split ---
split = {
    "train_idx": list(map(int, train_idx)),
    "val_idx": list(map(int, val_idx)),
    "test_idx": list(map(int, test_idx))
}

os.makedirs("splits", exist_ok=True)

with open("splits/policy_split.json", "w") as f:
    json.dump(split, f)

print("✅ Policy split saved!")

✅ Policy split saved!


### Graph Construction

In [6]:
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import pickle

# =========================
# Load dataset
# =========================
df = pd.read_csv("processed_dataset.csv")

# =========================
# Extract coordinates
# =========================
coords = df[['LATITUDE', 'LONGITUDE']].values
n = len(coords)

# =========================
# Build kNN graph
# =========================
k = 20

nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree')
nbrs.fit(coords)

distances, indices = nbrs.kneighbors(coords)

# Remove self-loops (first neighbor is itself)
distances = distances[:, 1:]
indices = indices[:, 1:]

# =========================
# Compute Gaussian weights
# =========================
# σ = average neighbor distance (robust default)
sigma = np.mean(distances)

weights = np.exp(-(distances ** 2) / (sigma ** 2))

# =========================
# Build sparse adjacency matrix
# =========================
rows = np.repeat(np.arange(n), k)
cols = indices.flatten()
vals = weights.flatten()

A = sp.coo_matrix((vals, (rows, cols)), shape=(n, n))

# =========================
# Symmetrize graph (undirected)
# =========================
A = (A + A.T) / 2

# =========================
# Compute Laplacian
# =========================
D = sp.diags(A.sum(axis=1).A1)
L = D - A

# =========================
# Save graph
# =========================
graph_data = {
    "adjacency": A.tocsr(),   # efficient format
    "laplacian": L.tocsr(),
    "k": k,
    "sigma": float(sigma)
}

with open("graph.pkl", "wb") as f:
    pickle.dump(graph_data, f)

print("✅ Graph constructed and saved as graph.pkl")

✅ Graph constructed and saved as graph.pkl


In [7]:
print("Nodes:", n)
print("Edges (non-zero entries):", A.nnz)
print("k:", k)
print("sigma:", sigma)

# Check symmetry
print("Symmetry check (should be ~0):", (A - A.T).nnz)

# Weight range
print("Min weight:", weights.min())
print("Max weight:", weights.max())

Nodes: 9303
Edges (non-zero entries): 229100
k: 20
sigma: 0.014425909486433033
Symmetry check (should be ~0): 0
Min weight: 0.0
Max weight: 1.0


### Non-Graph Baseline Models 

In [8]:
from sklearn.linear_model import LogisticRegression
C_grid = [0.01, 0.1, 1, 10]

C = 1.0  # default value 
LR_mdoel = LogisticRegression(
    max_iter=1000,
    C = C,
    class_weight='balanced'  # important for imbalance
)

In [9]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # binary
        )

    def forward(self, x):
        return self.net(x)

In [10]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [11]:

import xgboost as xgb

# will be updated for each training regime 
imbalance_ratio = None 
# -------------------------
# Base XGBoost parameters (defined once)
# -------------------------
base_xgb_params = dict(
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
)

# -------------------------
# Hyperparameter grid
# -------------------------
max_depth_grid = [4, 6, 8]
lr_grid = [0.01, 0.05, 0.1]
n_estimators_grid = [200, 300]

## Training Baselines in Cross City OOD 

#### Logistic Regression Baseline Training 

In [42]:
from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
import json
import glob
from datetime import datetime

# -------------------------
# Config / Seeds
# -------------------------
seeds = [0, 1, 2, 3, 4]

param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0]
}

base_model_config = {
    "model_type": "LogisticRegression",
    "max_iter": 1000,
    "class_weight": "balanced"
}

# -------------------------
# Load data
# -------------------------
df = pd.read_csv("processed_dataset.csv")

label_col = "label"
city_col = "CITY"

X_all = df.drop(columns=[label_col, city_col])
X_all = X_all.select_dtypes(include=[np.number])
y_all = df[label_col].values

# -------------------------
# Metric storage
# -------------------------
mean_f1_list = []
worst_f1_list = []
mean_acc_list = []
worst_acc_list = []

# -------------------------
# JSON logging
# -------------------------
experiment_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "base_model_config": base_model_config,
    "param_grid": param_grid,
    "seeds": []
}

# -------------------------
# Run experiment
# -------------------------
for seed in seeds:
    print(f"\n===== Running seed {seed} =====")

    split_files = sorted(glob.glob(f"splits/city_seed{seed}_split*.json"))

    env_results = []

    seed_log = {
        "seed": seed,
        "splits": [],
        "summary": {}
    }

    for split_file in split_files:
        with open(split_file, "r") as f:
            split = json.load(f)

        train_idx = split["train_idx"]
        val_idx = split["val_idx"]
        test_idx = split["test_idx"]
        test_city = split["test_city"]

        # -------------------------
        # Split data
        # -------------------------
        X_train = X_all.iloc[train_idx]
        y_train = y_all[train_idx]

        X_val = X_all.iloc[val_idx]
        y_val = y_all[val_idx]

        X_test = X_all.iloc[test_idx]
        y_test = y_all[test_idx]

        # -------------------------
        # Hyperparameter tuning
        # -------------------------
        best_score = -1
        best_params = None

        for C in param_grid["C"]:
            model = LogisticRegression(
                C=C,
                max_iter=base_model_config["max_iter"],
                class_weight=base_model_config["class_weight"],
                random_state=seed
            )

            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)

            val_f1 = f1_score(y_val, val_pred, average="weighted")

            if val_f1 > best_score:
                best_score = val_f1
                best_params = {"C": C}

        # -------------------------
        # Retrain with best params
        # (strict version: train only on train set)
        # -------------------------
        best_model = LogisticRegression(
            C=best_params["C"],
            max_iter=base_model_config["max_iter"],
            class_weight=base_model_config["class_weight"],
            random_state=seed
        )

        best_model.fit(X_train, y_train)

        # -------------------------
        # Test evaluation
        # -------------------------
        y_pred = best_model.predict(X_test)

        f1 = f1_score(y_test, y_pred, average="weighted")
        acc = accuracy_score(y_test, y_pred)

        result_entry = {
            "city": test_city,
            "val_f1": float(best_score),
            "test_f1": float(f1),
            "test_acc": float(acc),
            "best_params": best_params
        }

        env_results.append(result_entry)
        seed_log["splits"].append(result_entry)

        print(f"{test_city} → ValF1: {best_score:.4f}, TestF1: {f1:.4f}, Acc: {acc:.4f}")

    # -------------------------
    # Aggregate per seed
    # -------------------------
    results_df = pd.DataFrame(env_results)

    mean_f1 = results_df["test_f1"].mean()
    worst_f1 = results_df["test_f1"].min()

    mean_acc = results_df["test_acc"].mean()
    worst_acc = results_df["test_acc"].min()

    mean_f1_list.append(mean_f1)
    worst_f1_list.append(worst_f1)
    mean_acc_list.append(mean_acc)
    worst_acc_list.append(worst_acc)

    seed_log["summary"] = {
        "mean_f1": float(mean_f1),
        "worst_f1": float(worst_f1),
        "mean_acc": float(mean_acc),
        "worst_acc": float(worst_acc)
    }

    experiment_log["seeds"].append(seed_log)

    print(f"\nSeed {seed} Summary:")
    print(f"MeanEnvF1:   {mean_f1:.4f}")
    print(f"WorstEnvF1:  {worst_f1:.4f}")
    print(f"MeanEnvAcc:  {mean_acc:.4f}")
    print(f"WorstEnvAcc: {worst_acc:.4f}")

# -------------------------
# Final aggregation
# -------------------------
final_results_LR = {
    "mean_f1_mean": float(np.mean(mean_f1_list)),
    "mean_f1_std": float(np.std(mean_f1_list)),
    "worst_f1_mean": float(np.mean(worst_f1_list)),
    "worst_f1_std": float(np.std(worst_f1_list)),
    "mean_acc_mean": float(np.mean(mean_acc_list)),
    "mean_acc_std": float(np.std(mean_acc_list)),
    "worst_acc_mean": float(np.mean(worst_acc_list)),
    "worst_acc_std": float(np.std(worst_acc_list)),
}

# -------------------------
# Final metrics table
# -------------------------
final_table_LR = pd.DataFrame([{
    "MeanEnvF1": f"{final_results_LR['mean_f1_mean']:.4f} ± {final_results_LR['mean_f1_std']:.4f}",
    "WorstEnvF1": f"{final_results_LR['worst_f1_mean']:.4f} ± {final_results_LR['worst_f1_std']:.4f}",
    "MeanEnvAcc": f"{final_results_LR['mean_acc_mean']:.4f} ± {final_results_LR['mean_acc_std']:.4f}",
    "WorstEnvAcc": f"{final_results_LR['worst_acc_mean']:.4f} ± {final_results_LR['worst_acc_std']:.4f}"
}])

print("\n===== Final Metrics Table =====")
print(final_table_LR)

experiment_log["final_results"] = final_results_LR

print("\n===== FINAL RESULTS (Across Seeds) =====")
print(f"MeanEnvF1:   {final_results_LR['mean_f1_mean']:.4f} ± {final_results_LR['mean_f1_std']:.4f}")
print(f"WorstEnvF1:  {final_results_LR['worst_f1_mean']:.4f} ± {final_results_LR['worst_f1_std']:.4f}")
print(f"MeanEnvAcc:  {final_results_LR['mean_acc_mean']:.4f} ± {final_results_LR['mean_acc_std']:.4f}")
print(f"WorstEnvAcc: {final_results_LR['worst_acc_mean']:.4f} ± {final_results_LR['worst_acc_std']:.4f}")

# -------------------------
# Save JSON log
# -------------------------
with open("experiment_results.json", "w") as f:
    json.dump(experiment_log, f, indent=4)

print("\nSaved experiment log to experiment_results_LR_city.json")


===== Running seed 0 =====
UNINCORPORATED → ValF1: 0.0854, TestF1: 0.2699, Acc: 0.4410
SAN BERNARDINO → ValF1: 0.1070, TestF1: 0.0915, Acc: 0.2380
RANCHO CUCAMONGA → ValF1: 0.1076, TestF1: 0.0472, Acc: 0.1659
CHINO → ValF1: 0.1076, TestF1: 0.0312, Acc: 0.1329
MONTCLAIR → ValF1: 0.1069, TestF1: 0.0421, Acc: 0.1561
YUCCA VALLEY → ValF1: 0.1037, TestF1: 0.2209, Acc: 0.3922
HESPERIA → ValF1: 0.1032, TestF1: 0.1341, Acc: 0.2947
UPLAND → ValF1: 0.1083, TestF1: 0.0334, Acc: 0.1378
ADELANTO → ValF1: 0.1038, TestF1: 0.1470, Acc: 0.3103
BARSTOW → ValF1: 0.1018, TestF1: 0.2695, Acc: 0.4406
BIG BEAR LAKE → ValF1: 0.1040, TestF1: 0.0725, Acc: 0.2093
APPLE VALLEY → ValF1: 0.1020, TestF1: 0.2190, Acc: 0.3901
COLTON → ValF1: 0.1037, TestF1: 0.1153, Acc: 0.2707
CHINO HILLS → ValF1: 0.1053, TestF1: 0.0404, Acc: 0.1527
TWENTYNINE PALMS → ValF1: 0.1034, TestF1: 0.3903, Acc: 0.5500
LOMA LINDA → ValF1: 0.1039, TestF1: 0.1402, Acc: 0.3021
GRAND TERRACE → ValF1: 0.1042, TestF1: 0.0278, Acc: 0.1250
NEEDLES → 

### MLP on Cross City 

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim

In [28]:
def train_model(model, X_train, y_train, X_val, y_val, lr, epochs=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    X_train = torch.tensor(X_train.values, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

    X_val = torch.tensor(X_val.values, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1).to(device)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    # --- Validation ---
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_preds = (val_probs > 0.5).astype(int)

    val_f1 = f1_score(y_val.cpu().numpy(), val_preds, average="weighted")
    return model, val_f1

In [26]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Ensures reproducibility (but can reduce randomness)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [45]:
import json
import glob
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score

# -------------------------
# Config
# -------------------------
lr_grid = [1e-4, 1e-3, 1e-2]
seeds = [0, 1, 2, 3, 4]

base_model_config = {
    "model_type": "MLP",
    "lr_grid": lr_grid
}

# -------------------------
# Metric storage
# -------------------------
mean_f1_list = []
worst_f1_list = []
mean_acc_list = []
worst_acc_list = []

# -------------------------
# JSON logging
# -------------------------
experiment_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_config": base_model_config,
    "seeds": []
}

# -------------------------
# Run experiment
# -------------------------
for seed in seeds:
    print(f"\n===== Running seed {seed} =====")

    set_seed(seed)

    split_files = sorted(glob.glob(f"splits/city_seed{seed}_split*.json"))

    env_results = []

    seed_log = {
        "seed": seed,
        "splits": [],
        "summary": {}
    }

    for split_file in split_files:
        with open(split_file, "r") as f:
            split = json.load(f)

        train_idx = split["train_idx"]
        val_idx = split["val_idx"]          # assuming available
        test_idx = split["test_idx"]
        test_city = split["test_city"]

        # -------------------------
        # Data
        # -------------------------
        X_train = X_all.iloc[train_idx]
        y_train = y_all[train_idx]

        X_val = X_all.iloc[val_idx]
        y_val = y_all[val_idx]

        X_test = X_all.iloc[test_idx]
        y_test = y_all[test_idx]

        input_dim = X_train.shape[1]

        # -------------------------
        # Hyperparameter tuning
        # -------------------------
        best_model = None
        best_f1 = -1
        best_lr = None

        lr_results = []

        for lr in lr_grid:
            model = MLP(input_dim)

            trained_model, val_f1 = train_model(
                model, X_train, y_train, X_val, y_val, lr
            )

            lr_results.append({
                "lr": lr,
                "val_f1": float(val_f1)
            })

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_model = trained_model
                best_lr = lr

        # -------------------------
        # Test evaluation
        # -------------------------
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        X_test_tensor = torch.tensor(
            X_test.values, dtype=torch.float32
        ).to(device)

        best_model.eval()
        with torch.no_grad():
            test_logits = best_model(X_test_tensor)
            test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()
            y_pred = (test_probs > 0.5).astype(int)

        f1 = f1_score(y_test, y_pred, average="weighted")
        acc = accuracy_score(y_test, y_pred)

        # -------------------------
        # Logging per split
        # -------------------------
        result_entry = {
            "city": test_city,
            "val_f1": float(best_f1),
            "test_f1": float(f1),
            "test_acc": float(acc),
            "best_hyperparameters": {
                "lr": best_lr
            },
            "lr_search": lr_results
        }

        env_results.append(result_entry)
        seed_log["splits"].append(result_entry)

        print(
            f"{test_city} → ValF1: {best_f1:.4f}, "
            f"TestF1: {f1:.4f}, Acc: {acc:.4f}, BestLR: {best_lr}"
        )

    # -------------------------
    # Aggregate per seed
    # -------------------------
    results_df = pd.DataFrame(env_results)

    mean_f1 = results_df["test_f1"].mean()
    worst_f1 = results_df["test_f1"].min()

    mean_acc = results_df["test_acc"].mean()
    worst_acc = results_df["test_acc"].min()

    mean_f1_list.append(mean_f1)
    worst_f1_list.append(worst_f1)
    mean_acc_list.append(mean_acc)
    worst_acc_list.append(worst_acc)

    seed_log["summary"] = {
        "mean_f1": float(mean_f1),
        "worst_f1": float(worst_f1),
        "mean_acc": float(mean_acc),
        "worst_acc": float(worst_acc)
    }

    experiment_log["seeds"].append(seed_log)

    print(f"\nSeed {seed} Summary:")
    print(f"MeanEnvF1:   {mean_f1:.4f}")
    print(f"WorstEnvF1:  {worst_f1:.4f}")
    print(f"MeanEnvAcc:  {mean_acc:.4f}")
    print(f"WorstEnvAcc: {worst_acc:.4f}")

# -------------------------
# Final aggregation
# -------------------------
final_results = {
    "mean_f1_mean": float(np.mean(mean_f1_list)),
    "mean_f1_std": float(np.std(mean_f1_list)),
    "worst_f1_mean": float(np.mean(worst_f1_list)),
    "worst_f1_std": float(np.std(worst_f1_list)),
    "mean_acc_mean": float(np.mean(mean_acc_list)),
    "mean_acc_std": float(np.std(mean_acc_list)),
    "worst_acc_mean": float(np.mean(worst_acc_list)),
    "worst_acc_std": float(np.std(worst_acc_list)),
}

# -------------------------
# Final metrics table
# -------------------------
final_table_MLP = pd.DataFrame([{
    "MeanEnvF1": f"{final_results['mean_f1_mean']:.4f} ± {final_results['mean_f1_std']:.4f}",
    "WorstEnvF1": f"{final_results['worst_f1_mean']:.4f} ± {final_results['worst_f1_std']:.4f}",
    "MeanEnvAcc": f"{final_results['mean_acc_mean']:.4f} ± {final_results['mean_acc_std']:.4f}",
    "WorstEnvAcc": f"{final_results['worst_acc_mean']:.4f} ± {final_results['worst_acc_std']:.4f}"
}])

print("\n===== Final Metrics Table =====")
print(final_table_MLP)
experiment_log["final_results"] = final_results

print("\n===== FINAL MLP RESULTS (Across Seeds) =====")
print(f"MeanEnvF1:   {final_results['mean_f1_mean']:.4f} ± {final_results['mean_f1_std']:.4f}")
print(f"WorstEnvF1:  {final_results['worst_f1_mean']:.4f} ± {final_results['worst_f1_std']:.4f}")
print(f"MeanEnvAcc:  {final_results['mean_acc_mean']:.4f} ± {final_results['mean_acc_std']:.4f}")
print(f"WorstEnvAcc: {final_results['worst_acc_mean']:.4f} ± {final_results['worst_acc_std']:.4f}")

# -------------------------
# Save JSON
# -------------------------
with open("mlp_experiment_results.json", "w") as f:
    json.dump(experiment_log, f, indent=4)

print("\nSaved experiment log to mlp_experiment_results.json")


===== Running seed 0 =====
UNINCORPORATED → ValF1: 0.6711, TestF1: 0.4009, Acc: 0.5590, BestLR: 0.0001
SAN BERNARDINO → ValF1: 0.6300, TestF1: 0.6591, Acc: 0.7620, BestLR: 0.01
RANCHO CUCAMONGA → ValF1: 0.6290, TestF1: 0.7586, Acc: 0.8341, BestLR: 0.001
CHINO → ValF1: 0.6289, TestF1: 0.8053, Acc: 0.8671, BestLR: 0.0001
MONTCLAIR → ValF1: 0.6302, TestF1: 0.7725, Acc: 0.8439, BestLR: 0.0001
YUCCA VALLEY → ValF1: 0.6361, TestF1: 0.4596, Acc: 0.6078, BestLR: 0.0001
HESPERIA → ValF1: 0.6369, TestF1: 0.5835, Acc: 0.7053, BestLR: 0.0001
UPLAND → ValF1: 0.6277, TestF1: 0.7984, Acc: 0.8622, BestLR: 0.0001
ADELANTO → ValF1: 0.6358, TestF1: 0.5630, Acc: 0.6897, BestLR: 0.0001
BARSTOW → ValF1: 0.6394, TestF1: 0.4014, Acc: 0.5594, BestLR: 0.0001
BIG BEAR LAKE → ValF1: 0.6354, TestF1: 0.6983, Acc: 0.7907, BestLR: 0.0001
APPLE VALLEY → ValF1: 0.6392, TestF1: 0.4621, Acc: 0.6099, BestLR: 0.001
COLTON → ValF1: 0.6359, TestF1: 0.6152, Acc: 0.7293, BestLR: 0.0001
CHINO HILLS → ValF1: 0.6331, TestF1: 0.7

### XGB on Cross City 

In [20]:
param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300]
}

In [47]:
import json
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score

# -------------------------
# Config
# -------------------------
seeds = [0, 1, 2, 3, 4]

# -------------------------
# Storage for final stats
# -------------------------
mean_f1_list = []
worst_f1_list = []
mean_acc_list = []
worst_acc_list = []

# -------------------------
# JSON logging
# -------------------------
experiment_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_type": "XGBoost",
    "param_grid": {
        "max_depth": max_depth_grid,
        "learning_rate": lr_grid,
        "n_estimators": n_estimators_grid
    },
    "base_params": base_xgb_params,
    "seeds": [],
    "final_results": {}
}

# -------------------------
# Run experiment
# -------------------------
for seed in seeds:
    print(f"\n===== Running seed {seed} =====")

    np.random.seed(seed)

    split_files = sorted(glob.glob(f"splits/city_seed{seed}_split*.json"))

    env_results = []

    seed_log = {
        "seed": seed,
        "splits": [],
        "summary": {}
    }

    for split_file in split_files:
        with open(split_file, "r") as f:
            split = json.load(f)

        train_idx = split["train_idx"]
        val_idx = split["val_idx"]
        test_idx = split["test_idx"]
        test_city = split["test_city"]

        # -------------------------
        # Data
        # -------------------------
        X_train = X_all.iloc[train_idx]
        y_train = y_all[train_idx]

        X_val = X_all.iloc[val_idx]
        y_val = y_all[val_idx]

        X_test = X_all.iloc[test_idx]
        y_test = y_all[test_idx]

        # -------------------------
        # Class imbalance
        # -------------------------
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        scale_pos_weight = neg / pos if pos > 0 else 1

        # -------------------------
        # Hyperparameter tuning
        # -------------------------
        best_model = None
        best_f1 = -1
        best_params = None

        search_results = []

        for max_depth in max_depth_grid:
            for lr in lr_grid:
                for n_estimators in n_estimators_grid:

                    params = {
                        **base_xgb_params,
                        "max_depth": max_depth,
                        "learning_rate": lr,
                        "n_estimators": n_estimators,
                        "scale_pos_weight": scale_pos_weight,
                        "random_state": seed
                    }

                    model = xgb.XGBClassifier(**params)

                    model.fit(
                        X_train, y_train,
                        eval_set=[(X_val, y_val)],
                        verbose=False
                    )

                    y_val_pred = model.predict(X_val)
                    val_f1 = f1_score(y_val, y_val_pred, average="weighted")

                    search_results.append({
                        "params": {
                            "max_depth": max_depth,
                            "learning_rate": lr,
                            "n_estimators": n_estimators
                        },
                        "val_f1": float(val_f1)
                    })

                    if val_f1 > best_f1:
                        best_f1 = val_f1
                        best_model = model
                        best_params = {
                            "max_depth": max_depth,
                            "learning_rate": lr,
                            "n_estimators": n_estimators,
                            "scale_pos_weight": float(scale_pos_weight)
                        }

        # -------------------------
        # Test evaluation
        # -------------------------
        y_pred = best_model.predict(X_test)

        f1 = f1_score(y_test, y_pred, average="weighted")
        acc = accuracy_score(y_test, y_pred)

        result_entry = {
            "city": test_city,
            "val_f1": float(best_f1),
            "test_f1": float(f1),
            "test_acc": float(acc),
            "best_hyperparameters": best_params,
            "search_results": search_results
        }

        env_results.append(result_entry)
        seed_log["splits"].append(result_entry)

        print(f"{test_city} → ValF1: {best_f1:.4f}, TestF1: {f1:.4f}, Acc: {acc:.4f}")

    # -------------------------
    # Aggregate per seed
    # -------------------------
    results_df = pd.DataFrame(env_results)

    mean_f1 = results_df["test_f1"].mean()
    worst_f1 = results_df["test_f1"].min()
    mean_acc = results_df["test_acc"].mean()
    worst_acc = results_df["test_acc"].min()

    mean_f1_list.append(mean_f1)
    worst_f1_list.append(worst_f1)
    mean_acc_list.append(mean_acc)
    worst_acc_list.append(worst_acc)

    seed_log["summary"] = {
        "mean_f1": float(mean_f1),
        "worst_f1": float(worst_f1),
        "mean_acc": float(mean_acc),
        "worst_acc": float(worst_acc)
    }

    experiment_log["seeds"].append(seed_log)

# -------------------------
# Final aggregation across seeds
# -------------------------
final_results = {
    "mean_f1_mean": float(np.mean(mean_f1_list)),
    "mean_f1_std": float(np.std(mean_f1_list)),
    "worst_f1_mean": float(np.mean(worst_f1_list)),
    "worst_f1_std": float(np.std(worst_f1_list)),
    "mean_acc_mean": float(np.mean(mean_acc_list)),
    "mean_acc_std": float(np.std(mean_acc_list)),
    "worst_acc_mean": float(np.mean(worst_acc_list)),
    "worst_acc_std": float(np.std(worst_acc_list)),
}

experiment_log["final_results"] = final_results

print("\n===== FINAL RESULTS (Across Seeds) =====")
print(f"MeanEnvF1:   {final_results['mean_f1_mean']:.4f} ± {final_results['mean_f1_std']:.4f}")
print(f"WorstEnvF1:  {final_results['worst_f1_mean']:.4f} ± {final_results['worst_f1_std']:.4f}")
print(f"MeanEnvAcc:  {final_results['mean_acc_mean']:.4f} ± {final_results['mean_acc_std']:.4f}")
print(f"WorstEnvAcc: {final_results['worst_acc_mean']:.4f} ± {final_results['worst_acc_std']:.4f}")

# -------------------------
# Final metrics table
# -------------------------
final_table_XGB = pd.DataFrame([{
    "MeanEnvF1": f"{final_results['mean_f1_mean']:.4f} ± {final_results['mean_f1_std']:.4f}",
    "WorstEnvF1": f"{final_results['worst_f1_mean']:.4f} ± {final_results['worst_f1_std']:.4f}",
    "MeanEnvAcc": f"{final_results['mean_acc_mean']:.4f} ± {final_results['mean_acc_std']:.4f}",
    "WorstEnvAcc": f"{final_results['worst_acc_mean']:.4f} ± {final_results['worst_acc_std']:.4f}"
}])

print("\n===== Final Metrics Table =====")
print(final_table_XGB)

# Add to JSON
experiment_log["final_metrics_table"] = final_table_XGB.to_dict(orient="records")

# -------------------------
# Save JSON
# -------------------------
with open("xgb_experiment_results.json", "w") as f:
    json.dump(experiment_log, f, indent=4)

print("\nSaved results to xgb_experiment_results.json")


===== Running seed 0 =====
UNINCORPORATED → ValF1: 0.8006, TestF1: 0.5790, Acc: 0.5978
SAN BERNARDINO → ValF1: 0.7839, TestF1: 0.7519, Acc: 0.7368
RANCHO CUCAMONGA → ValF1: 0.7841, TestF1: 0.8638, Acc: 0.8610
CHINO → ValF1: 0.7640, TestF1: 0.8637, Acc: 0.8613
MONTCLAIR → ValF1: 0.7773, TestF1: 0.8271, Acc: 0.8376
YUCCA VALLEY → ValF1: 0.7740, TestF1: 0.8640, Acc: 0.8627
HESPERIA → ValF1: 0.7930, TestF1: 0.6977, Acc: 0.6865
UPLAND → ValF1: 0.7761, TestF1: 0.8810, Acc: 0.8822
ADELANTO → ValF1: 0.7676, TestF1: 0.7619, Acc: 0.7586
BARSTOW → ValF1: 0.7821, TestF1: 0.6227, Acc: 0.6224
BIG BEAR LAKE → ValF1: 0.7954, TestF1: 0.8451, Acc: 0.8372
APPLE VALLEY → ValF1: 0.7807, TestF1: 0.7232, Acc: 0.7198
COLTON → ValF1: 0.7761, TestF1: 0.8231, Acc: 0.8158
CHINO HILLS → ValF1: 0.7730, TestF1: 0.8487, Acc: 0.8702
TWENTYNINE PALMS → ValF1: 0.7850, TestF1: 0.6509, Acc: 0.6500
LOMA LINDA → ValF1: 0.7940, TestF1: 0.7934, Acc: 0.8021
GRAND TERRACE → ValF1: 0.7880, TestF1: 0.8184, Acc: 0.7917
NEEDLES → 

## Need to update each and how they save results so that I can make a table at the end 

# Summary of Baseline Models

In [49]:
summary = pd.DataFrame({
    "Model": ["Logistic Regression", "MLP", "XGBoost"],

    "MeanEnvF1": [
        final_table_LR["MeanEnvF1"].iloc[0],
        final_table_MLP["MeanEnvF1"].iloc[0],
        final_table_XGB["MeanEnvF1"].iloc[0]
    ],

    "WorstEnvF1": [
        final_table_LR["WorstEnvF1"].iloc[0],
        final_table_MLP["WorstEnvF1"].iloc[0],
        final_table_XGB["WorstEnvF1"].iloc[0]
    ],

    "MeanEnvAcc": [
        final_table_LR["MeanEnvAcc"].iloc[0],
        final_table_MLP["MeanEnvAcc"].iloc[0],
        final_table_XGB["MeanEnvAcc"].iloc[0]
    ],

    "WorstEnvAcc": [
        final_table_LR["WorstEnvAcc"].iloc[0],
        final_table_MLP["WorstEnvAcc"].iloc[0],
        final_table_XGB["WorstEnvAcc"].iloc[0]
    ]
})

print("\n===== Model Comparison Table =====")
print(summary)


===== Model Comparison Table =====
                 Model        MeanEnvF1       WorstEnvF1       MeanEnvAcc   
0  Logistic Regression  0.1245 ± 0.0028  0.0278 ± 0.0000  0.2671 ± 0.0011  \
1                  MLP  0.6222 ± 0.0062  0.2466 ± 0.0653  0.7306 ± 0.0057   
2              XGBoost  0.7750 ± 0.0010  0.5681 ± 0.0393  0.7718 ± 0.0015   

       WorstEnvAcc  
0  0.1250 ± 0.0000  
1  0.4143 ± 0.0714  
2  0.5571 ± 0.0506  
